# 01 — Text Preprocessing

Educational notebook for **NOVA Phase 2**. These experiments run the *same*
functions the API uses (`backend/app/nlp/`), interactively.

**What you will explore:**

1. Cleaning & normalization (why `Hello` ≠ `hello` to a counter)
2. Tokenization — naive regex vs NLTK rule-based
3. Stopwords — and why removing `not` can invert meaning
4. Stemming (Porter rules) vs lemmatization (dictionary lookup)
5. Switching pipeline steps on/off with `PreprocessConfig`

> Run top-to-bottom. The setup cell downloads ~2 MB of NLTK corpora once.
> Docs: `docs/nlp/01_preprocessing.md`, `docs/nlp/02_tokenization.md`

In [ ]:
import re
import sys
from pathlib import Path

# Make the backend package importable from notebooks/
cwd = Path.cwd()
backend_dir = cwd.parent / "backend" if cwd.name == "notebooks" else cwd / "backend"
sys.path.insert(0, str(backend_dir))

from app.nlp.nltk_data import ensure_nltk_data
print("NLTK data:", ensure_nltk_data() or "already present")

from app.nlp.preprocessing import (
    PreprocessConfig, clean_text, get_stopwords, lemmatize_tokens,
    preprocess, remove_stopwords, stem_tokens,
)
from app.nlp.tokenizer import naive_word_tokenize, sentence_tokenize, word_tokenize

print("imports OK")

## 1. Raw material

Real chat messages are messy: mixed case, URLs, HTML, punctuation noise.

In [ ]:
messages = [
    "Hello, how are you?",
    "Check <b>this</b> link: https://example.com NOW!",
    "The cats are sleeping... aren't they?",
    "I DON'T like extra    whitespace",
]
for m in messages:
    print(repr(m))

## 2. Cleaning & normalization

**Why:** `Hello` and `hello` are the same word to us — to a counter they are
two different rows in the vocabulary. URLs and HTML are pure noise.

Cleaning is a **many-to-one mapping**: many surface forms → one canonical form.
It shrinks the vocabulary *before* any math happens.

In [ ]:
config = PreprocessConfig()  # defaults: clean + lowercase ON

for m in messages:
    print(f"{m!r}\n  -> {clean_text(m, config)!r}\n")

### When NOT to lowercase

Case can be signal: `US` vs `us`, `Amazon` (company) vs `amazon` (river).
Transformers are trained on *cased* natural text — we lowercase only for
classical bag-of-words methods.

In [ ]:
print(clean_text("US markets vs us markets", PreprocessConfig(lowercase=False)))
print(clean_text("US markets vs us markets", PreprocessConfig(lowercase=True)))

## 3. Tokenization — naive vs rule-based

Start with the simplest tokenizer you could possibly write, then see what
you pay for it.

In [ ]:
text = "I don't know... Dr. Smith said 3.14 is fine!"
print("naive:", naive_word_tokenize(text))
print("nltk :", word_tokenize(text))

In [ ]:
# What the naive version loses:
# - punctuation (question vs statement signal)
# - contraction structure (don't should be do + n't)
# - any non-Latin script (regex is [A-Za-z0-9']+)
print("naive:", naive_word_tokenize("\u4e2d\u6587 words \u00e9l\u00e8ve"))
print("nltk :", word_tokenize("Chinese words \u00e9l\u00e8ve"))

### Sentence tokenization

A regex split after `.`/`?`/`!` fails on abbreviations. Punkt learned from
data which periods do **not** end sentences.

In [ ]:
text = "Dr. Smith arrived. He was late! Was the meeting good?"
print("punkt :", sentence_tokenize(text))
print("regex :", [p for p in re.split(r"(?<=[.!?])\s+", text) if p])

## 4. Stopwords

High-frequency function words appear in *every* document, so they carry
little topical signal in bag-of-words models.

**The trap:** NLTK's English list includes `not` — removing it destroys negation.

In [ ]:
tokens = word_tokenize("the cat and the dog are not happy")
sw = get_stopwords()
print("tokens  :", tokens)
print("kept    :", [t for t in tokens if t not in sw])
print("removed :", [t for t in tokens if t in sw])
print()
print("'not happy' would become 'happy' if we trust the list blindly!")

### Contraction artifact (a real pipeline quirk)

In [ ]:
tokens = word_tokenize("I don't know why")
kept = remove_stopwords(tokens)
print("tokens:", tokens)
print("kept  :", kept)
print("'n't' survives stopword removal because the list stores \"don't\", not \"n't\".")

## 5. Stemming (Porter)

Rule-based suffix chopping: ~150 ordered rewrite rules. Fast, works on
unknown words — but **lossy**: output is a pseudo-word, and it over-stems.

In [ ]:
for w in ["studies", "study", "cats", "running", "generously", "happily"]:
    print(f"{w:12} -> {stem_tokens([w])[0]}")

## 6. Lemmatization (WordNet)

Dictionary-aware: returns a real word (**lemma**). Part-of-speech matters —
the default is *noun*, so verbs slip through untouched.

In [ ]:
from nltk.stem import WordNetLemmatizer

for w in ["studies", "cats", "running", "better"]:
    print(f"{w:10} default -> {lemmatize_tokens([w])[0]}")

lem = WordNetLemmatizer()
print("running pos='v' ->", lem.lemmatize("running", pos="v"))
print("better  pos='a' ->", lem.lemmatize("better", pos="a"))

### Stem vs lemma side by side

In [ ]:
words = ["studies", "running", "cats", "generously"]
print(f"{'word':12} {'stem':10} {'lemma':10}")
for w in words:
    print(f"{w:12} {stem_tokens([w])[0]:10} {lemmatize_tokens([w])[0]:10}")

## 7. The full pipeline, one switch at a time

Every technique is optional — the *right* preprocessing depends on the model
that consumes it (see `docs/nlp/01_preprocessing.md` §6).

In [ ]:
text = "The cats are sleeping! Visit https://example.com for more info."

variants = {
    "all off (light clean)": PreprocessConfig(remove_punctuation=False, remove_stopwords=False),
    "default (punct+stop)":  PreprocessConfig(),
    "stemming":              PreprocessConfig(stemming=True),
    "lemmatization":         PreprocessConfig(lemmatization=True),
}

for name, cfg in variants.items():
    r = preprocess(text, cfg)
    print(f"{name:24} -> {r.tokens}")

In [ ]:
# Stemming + lemmatization are mutually exclusive (two normalizations fighting)
from pydantic import ValidationError
try:
    PreprocessConfig(stemming=True, lemmatization=True)
except ValidationError as exc:
    print("ValidationError:", str(exc).splitlines()[-1])

# Punctuation-only input legitimately yields zero content tokens
print(preprocess("!!! ???").tokens)

## 8. When NOT to preprocess

| Consuming model | Recipe |
|---|---|
| TF-IDF intent classifier (Phase 3) | everything ON |
| spaCy NER (Phase 4) | light cleaning only — spaCy tokenizes itself |
| Sentence embeddings (Phase 5) | light cleaning only, keep natural sentences |
| Transformers / LLMs (Phase 8) | none — they tokenize themselves |

**Takeaway:** preprocessing is a *model-specific design choice*, not a ritual.
Heavy cleaning helps bag-of-words and **hurts** neural models.

**Next:** `02_tf_idf.ipynb` — building TF-IDF by hand.